In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 1.5 Coupled Oscillators and Normal Modes

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume I — Elementary Mechanics",
    number="1.5",
    title="Coupled Oscillators and Normal Modes",
    blurb="Diagonalising the equations of motion: normal modes as an "
    "eigenvalue problem, the 1-D chain dispersion relation, and "
    "beating as two modes seen in the time domain.",
    difficulty="intermediate",
    estimate="60–90 min",
)

## Notebook overview

A single mass on a spring oscillates at one frequency. Couple a few of them
together and the motion looks hopelessly tangled: each mass pushes its
neighbours, which push back. The classical insight is that the tangle is an
illusion of the *coordinates*: in the right basis the system falls apart into
independent oscillators called **normal modes**, each a single frequency and a
fixed shape. Finding that basis is nothing more than solving an eigenvalue
problem, and once we have it the whole of small-oscillation theory (mode
shapes, the dispersion relation of a chain, even the slow beating of two weakly
coupled pendulums) follows from linear algebra.

We will (1) assemble the stiffness matrix and solve the mode eigenproblem,
(2) read the in-phase and anti-phase mode shapes off the eigenvectors, (3, 4)
check the mode frequencies of a three-mass system and a twenty-mass chain
against the analytic dispersion relation, (5) integrate the motion in time and
use energy conservation as an integrator check, (6) animate a single normal
mode oscillating rigidly at one frequency, and (7) animate the beating of two
weakly coupled oscillators and recover the beat period.

> **How to read the checks.** Each exercise ends with a validation that
> compares our result to an expected physical fact. A ✗ does *not* by itself
> mean the answer is wrong: it means the output didn't match what the check
> expected, which may be a real error, a different-but-valid convention (a
> sign, a unit, an array order), or simply too tight a tolerance. Treat a ✗ as
> a prompt to locate the discrepancy; passing is strong evidence of
> correctness, not proof.

> **Scope.** This is a working review, not a textbook chapter. For the theory
> of small oscillations and normal coordinates see Nolting, *Theoretical
> Physics 1–2* {cite}`nolting1,nolting2`, and Goldstein, Poole & Safko,
> *Classical Mechanics* {cite}`goldstein`.

## Theory in brief

### Small oscillations are linear

Take $N$ masses connected by springs and displaced slightly from equilibrium by
the coordinates $\mathbf x = (x_1,\dots,x_N)$. Expanding the potential to second
order, the equations of motion are linear and can be written in matrix form

```{math}
:label: eq-matrix-eom
\mathsf M\,\ddot{\mathbf x} = -\,\mathsf K\,\mathbf x,
```

with the **mass matrix** $\mathsf M$ (here $\mathsf M = m\,\mathsf I$) and the
**stiffness matrix** $\mathsf K$, whose entries are the second derivatives of
the potential at equilibrium. For a chain of equal springs $k$ between equal
masses, $\mathsf K$ is tridiagonal: $2k$ on the diagonal, $-k$ off it.

### Normal modes are an eigenvalue problem

Seeking oscillating solutions $\mathbf x(t) = \mathbf v\,e^{i\omega t}$ and
substituting into {eq}`eq-matrix-eom` turns the differential equation into the
**generalised eigenvalue problem**

```{math}
:label: eq-eig
\mathsf K\,\mathbf v = \omega^2\,\mathsf M\,\mathbf v.
```

Each eigenvector $\mathbf v$ is a **normal mode** (a fixed spatial pattern),
and the square root of its eigenvalue is the mode's frequency. The modes are
orthogonal with respect to $\mathsf M$, so any motion is a superposition of
modes that never exchange energy. This is the payoff: solving {eq}`eq-eig`
*diagonalises* the dynamics.

### Two masses between fixed walls

The smallest nontrivial case (two equal masses $m$, three springs $k$, fixed
walls at both ends) has stiffness matrix $\mathsf K = k\begin{pmatrix} 2 & -1
\\ -1 & 2 \end{pmatrix}$ and the two analytic modes

```{math}
:label: eq-n2
\omega_- = \sqrt{k/m}\quad\text{(in-phase)}, \qquad
\omega_+ = \sqrt{3k/m}\quad\text{(anti-phase)}.
```

### A chain has a dispersion relation

For $N$ equal masses in a fixed–fixed chain the eigenproblem {eq}`eq-eig` can be
solved in closed form: the standing-wave ansatz
$v_n \propto \sin\bigl(nj\pi/(N+1)\bigr)$, built to vanish at the two walls,
satisfies every tridiagonal row at once and gives the **discrete dispersion
relation**

```{math}
:label: eq-dispersion
\omega_j = 2\sqrt{k/m}\;\sin\!\left(\frac{j\pi}{2(N+1)}\right),
\qquad j = 1,\dots,N.
```

For small $j$ (long wavelength) this is linear in $j$ (the discrete echo of the
wave equation's $\omega = c\,q$, with $q$ the wavenumber) and it bends over
toward the band edge.

### Beating: two modes in the time domain

Couple two identical oscillators (each tied to a wall by a spring $k$) weakly to
each other by a spring $\kappa \ll k$. Their two modes sit at nearly equal
frequencies $\omega_-$ and $\omega_+$, and starting the system with all the
energy in one oscillator makes that energy slosh entirely into the other and
back, with **beat period**

```{math}
:label: eq-beat
T_\mathrm{beat} = \frac{2\pi}{\omega_+ - \omega_-},
```

the first complete transfer happening at $T_\mathrm{beat}/2$. Beating is not a
new phenomenon: it is just the two normal modes, with their slightly different
frequencies, drifting in and out of phase.

### The mass–spring chain

The system behind every exercise is a row of equal masses $m$ joined by equal
springs $k$, anchored to fixed walls at both ends: the fixed–fixed chain whose
stiffness matrix {eq}`eq-matrix-eom` is tridiagonal. The schematic shows the
smallest nontrivial case, $N=2$ (two masses, three springs), whose two analytic
modes are {eq}`eq-n2`; longer chains just repeat the unit cell.

In [ ]:
# (solution hidden on the public site)


---
## Setup

In [ ]:
import numpy as np
from scipy.linalg import eigh
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

from ecp import validate
from ecp.animate import show

# Unit masses and unit spring constant unless noted, so frequencies come out in
# units of sqrt(k/m). The mass matrix is then the identity.
M_MASS = 1.0
K_SPRING = 1.0


def stiffness(N, k=K_SPRING):
    """Stiffness matrix of a fixed-fixed chain of equal masses and springs.

    The tridiagonal K whose eigenproblem K v = ω^2 m v (eq-eig in the theory
    section) gives the normal modes: 2k on the diagonal, −k off it.

    Parameters
    ----------
    N : int
        Number of masses.
    k : float, optional
        Spring constant.

    Returns
    -------
    numpy.ndarray
        The N×N stiffness matrix.
    """
    K = np.zeros((N, N))
    np.fill_diagonal(K, 2.0 * k)
    idx = np.arange(N - 1)
    K[idx, idx + 1] = -k
    K[idx + 1, idx] = -k
    return K


def rhs(t, s, K):
    """First-order form of M d^2x/dt^2 = −K x with M = I.

    Parameters
    ----------
    t : float
        Time (unused).
    s : array_like
        State ``[x..., v...]``.
    K : numpy.ndarray
        Stiffness matrix.

    Returns
    -------
    numpy.ndarray
        The state derivative.
    """
    N = K.shape[0]
    x, v = s[:N], s[N:]
    return np.concatenate([v, -(K @ x) / M_MASS])


def energy(s, K):
    """Total mechanical energy v·v/2 + x·K·x/2 (with M = I).

    Parameters
    ----------
    s : array_like
        State ``[x..., v...]``.
    K : numpy.ndarray
        Stiffness matrix.

    Returns
    -------
    float
        The total energy (an integrator check).
    """
    N = K.shape[0]
    x, v = s[:N], s[N:]
    return 0.5 * M_MASS * v @ v + 0.5 * x @ K @ x

## Exercise 1 — Assemble the stiffness matrix and solve the eigenproblem

Everything starts from the matrix equation of motion {eq}`eq-matrix-eom`. Because the
system is linear, looking for single-frequency solutions collapses it to the
generalised eigenvalue problem {eq}`eq-eig`: the eigenvectors are the normal
modes and the square roots of the eigenvalues are their frequencies. `eigh`
solves exactly this, returning the eigenvalues in ascending order.

1. Build $\mathsf K$ for $N=2$ (two masses, fixed walls) with `stiffness`.
2. Solve $\mathsf K\,\mathbf v = \omega^2 \mathsf M\,\mathbf v$ with
   `scipy.linalg.eigh(K, M)`, take the square roots of the eigenvalues
   (`numpy.sqrt`), and return them sorted. They should be the analytic pair
   of {eq}`eq-n2`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    np.sort(w2),
    [1.0, np.sqrt(3.0)],
    "N=2 mode frequencies are √(k/m), √(3k/m)",
    rtol=1e-10,
)

## Exercise 2 — Interpret the mode shapes

The eigenvectors carry the *shape* of each mode. For the two-mass system
{eq}`eq-n2` predicts a low-frequency mode where the masses move together
(in-phase, the spring between them barely stretches) and a high-frequency mode
where they move oppositely (anti-phase, the middle spring does the most work).
The sign pattern of the eigenvector components is what distinguishes them.

Take the two eigenvectors from Exercise 1.

1. Identify the in-phase mode (lowest frequency, $\omega_-$) and the
   anti-phase mode (highest, $\omega_+$) by sorting on frequency
   (`numpy.argsort`).
2. Plot the two mode shapes as signed bars and confirm the in-phase mode has
   equal-sign components while the anti-phase mode has opposite signs.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    v_low[0] * v_low[1] > 0 and v_high[0] * v_high[1] < 0,
    "ω₋ is in-phase, ω₊ is anti-phase",
    f"in-phase product={v_low[0]*v_low[1]:+.3f}, "
    f"anti-phase product={v_high[0]*v_high[1]:+.3f}",
)

## Exercise 3 — Three masses: frequencies vs. the dispersion relation

The same machinery scales to any $N$. For a fixed–fixed chain the spectrum is
not arbitrary: it is fixed by the dispersion relation {eq}`eq-dispersion`. The
three-mass chain is the first case where the formula has an interior mode to
get right.

**Your task.** Build the $N=3$ chain, solve for its frequencies, and compare
them to $\omega_j = 2\sin(j\pi/8)$, $j=1,2,3$ from {eq}`eq-dispersion`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    np.sort(w3),
    disp3,
    "N=3 frequencies match 2 sin(jπ/8)",
    rtol=1e-10,
)

## Exercise 4 — The 1-D chain dispersion relation

Push $N$ up and the discrete frequencies trace out the full dispersion curve of
{eq}`eq-dispersion`: nearly linear for the long-wavelength (small-$j$) modes,
bending over to a flat band edge near $j=N$. Plotting the numeric frequencies
on top of the analytic sine is the clearest single picture of normal-mode
theory.

1. Build the $N=20$ chain and solve for all twenty frequencies (`stiffness`
   and the `mode_frequencies` helper).
2. Plot $\omega_j$ against the mode index $j$ and overlay the analytic
   dispersion curve of {eq}`eq-dispersion`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    wN,
    disp_N,
    "chain spectrum matches the dispersion relation",
    rtol=1e-9,
)

## Exercise 5 — Time evolution conserves energy

Frequencies and shapes come from the static eigenproblem; to *watch* the
system move we integrate {eq}`eq-matrix-eom` in time. A linear conservative system has
no business gaining or losing energy, so the relative drift of the total
energy is a direct check that the integrator is faithful: the same role
conservation laws played for the orbit in [§1.4](kepler-orbits.ipynb).

**Your task.** Pluck one mass of the $N=2$ system (displace it, the other at
rest), integrate with `scipy.integrate.solve_ivp` (`DOP853`), and confirm the
total energy $E=\tfrac12\mathbf v\cdot\mathbf v+\tfrac12\,\mathbf x^{\mathsf T}\mathsf K\mathbf x$
is conserved as the motion plays out.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.conserved(E, "total energy conserved (integrator check)", rel_drift=1e-6)

## Exercise 6 — A normal mode oscillates rigidly (worked animation)

Here is the defining property of a normal mode, made visible. Start the system
*exactly* in one eigenvector and every mass oscillates at the single frequency
$\omega_j$ of {eq}`eq-eig`, the spatial shape frozen: it only breathes up and
down. No energy leaks into any other mode. This is the worked example for the
two-animation rule; you build the second one in Exercise 7.

The animation shows the displacement profile of an $N=8$ chain initialised in
one mode, scaling rigidly in time.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    max_other_mode_amplitude,
    0.0,
    "a pure normal mode does not excite the other modes",
    atol=1e-6,
)

## Exercise 7 — Beating (student-implemented animation)

Two identical oscillators tied to walls and coupled weakly to each other have
two almost-equal mode frequencies. Start all the energy in one of them and the
two modes (initially in phase) slowly drift apart and back, sloshing the
energy entirely into the second oscillator and returning it, with the beat
period {eq}`eq-beat`. The first *complete* transfer happens at
$T_\mathrm{beat}/2$. Beating is the time-domain face of the two normal modes.

1. Build the two-oscillator system with wall springs $k=1$ and a weak
   coupling spring $\kappa = 0.05$, find its mode frequencies $\omega_\pm$
   with `scipy.linalg.eigh`, and integrate from $x_1=1$, $x_2=0$ (both at
   rest) over roughly one beat period with `scipy.integrate.solve_ivp`
   (`DOP853`).
2. **Build the animation** of the two masses sloshing energy back and forth:
   you have the trajectory `x1(t)`, `x2(t)` below; assemble a `FuncAnimation`
   of the two masses on their springs, `plt.close(fig)`, then display it with
   `ecp.animate.show(anim)`.
3. Measure the first full-transfer time from the *slow envelope* of mass 2's
   energy (a moving average via `numpy.convolve`) and compare it to
   $T_\mathrm{beat}/2 = \pi/(\omega_+-\omega_-)$ from {eq}`eq-beat`.

> A ✗ on the final check is almost always about the **measurement**, not the
> physics: a raw `argmax` on mass 2's fast-oscillating energy grabs a later
> revival. Smooth to the slow envelope (a moving average over about one fast
> period) before locating the first maximum.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    t_transfer,
    np.pi / (wplus - wminus),
    "energy fully transfers at T_beat/2",
    rtol=5e-3,
)

## Notebook summary

- The stiffness matrix and the eigenproblem $K\mathbf v=\omega^2 m\mathbf v$ for the normal
  modes; the in-phase and out-of-phase mode shapes; and three masses checked against the
  monatomic-chain dispersion $\omega=2\sqrt{k/m}\,|\sin(qa/2)|$.
- Time evolution conserving energy; a single normal mode oscillating rigidly at one frequency;
  and **beating** as the superposition of two near-degenerate modes.

## Outlook

- **Unequal masses or springs.** Break the symmetry and watch the mode shapes
  distort: the heavier mass moves less in the high-frequency mode.
- **The continuum limit.** As $N\to\infty$ the chain's dispersion relation
  {eq}`eq-dispersion` becomes the wave equation's $\omega = c\,q$ for long
  wavelengths; compare the small-$j$ slope to the sound speed.
- **Free–free boundaries.** Remove the walls and a zero-frequency mode appears,
  the uniform translation of the whole chain.
- **Forced and damped.** Drive one end and add damping; resonances appear
  exactly at the normal-mode frequencies found here.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()